# Misc: Parsing News Articles with `newspaper`

For **articles/news**, there is a library that feels almost magical: give it a URL → get the **title, clean
body, author, date**, without writing selectors one by one. This library uses
heuristics to **guess the main content** and discard menus/ads/footers (boilerplate removal).
Because of this, **the same code works across many news sites** (multi-source).

> **Installation note:** `newspaper3k` (the old one) often breaks on newer Python/lxml versions.
> We use its actively maintained successor: **`newspaper4k`** — the API is **the same**
> (`from newspaper import Article`).

Install:

```bash
uv sync --extra news        # adds newspaper4k
```

**Tooling:** `newspaper4k`. (Other alternatives: `trafilatura`, `news-please`.)

> ⚠️ This is **specifically for text articles**. For e-commerce (product/price grids) still use
> BeautifulSoup / XPath / API / Selenium.


## 1. Basic usage

The core pattern is **3 lines**:

```python
a = Article(url)
a.download()   # fetch the HTML
a.parse()      # extract title, text, author, date
```

Below we try it on a real URL. If the network/site blocks us, we fall back to
**sample HTML** via `set_html()` so the demo still works (deterministic).


In [1]:
from newspaper import Article

# Sample HTML (fallback) — a simple news article
SAMPLE_HTML = """
<html><head><title>Coffee Prices Rise 2026 - Economic News</title></head>
<body>
  <nav>Home | Economy | Sports | Technology</nav>
  <article>
    <h1>Coffee Prices Projected to Rise in 2026</h1>
    <p class="byline">By Andi Wijaya</p>
    <p>Global coffee prices are expected to rise this year due to extreme weather in several
       major producing countries.</p>
    <p>Farmers in some regions report a fairly significant drop in harvest yields
       compared to last year.</p>
    <p>Analysts expect this upward trend to continue through the end of the year.</p>
  </article>
  <footer>Copyright 2026 - All rights reserved.</footer>
</body></html>
"""


def parse_article(url, fallback_html=None):
    a = Article(url, language="en")
    try:
        a.download()
        a.parse()
        if not a.text and fallback_html:  # download succeeded but text is empty
            raise ValueError("empty text")
    except Exception as e:
        if fallback_html is None:
            raise
        print(f"(download failed: {e} -> using sample HTML)")
        a = Article(url, language="en")
        a.set_html(fallback_html)
        a.parse()
    return a


# try a real URL (Wikipedia: stable & does not block); fall back to sample HTML
art = parse_article("https://en.wikipedia.org/wiki/Web_scraping", fallback_html=SAMPLE_HTML)

print("TITLE  :", art.title)
print("AUTHOR :", art.authors)
print("DATE   :", art.publish_date)
print("TEXT   :", art.text[:300].replace("\n", " "), "...")


TITLE  : Web scraping
AUTHOR : ['Contributors to Wikimedia projects']
DATE   : 2005-09-17 18:57:30+00:00
TEXT   : For broader coverage of this topic, see Data scraping.  Method of extracting data from websites  "Web scraper" redirects here. For websites that scrape content, see Scraper site.  Web scraping, web harvesting, or web data extraction is data scraping used for extracting data from websites.[1] Web scr ...


## 2. Multi-source — the same code for many sites

This is the key advantage: **selectors don't need to change** per site. Just swap the URL.
Below we loop over several URLs; for each one that succeeds we show the title + text length.

(In the real world this could contain URLs from many different news sites.)


In [2]:
urls = [
    "https://en.wikipedia.org/wiki/Web_scraping",
    "https://en.wikipedia.org/wiki/Data_engineering",
    "https://en.wikipedia.org/wiki/Beautiful_Soup_(HTML_parser)",
]

for url in urls:
    try:
        a = Article(url)
        a.download()
        a.parse()
        print(f"[OK]     {a.title[:55]:55} | {len(a.text):>6} characters")
    except Exception as e:
        print(f"[FAILED] {url} -> {e}")


[OK]     Web scraping                                            |  20386 characters


[OK]     Data engineering                                        |   8170 characters


[OK]     Beautiful Soup (HTML parser)                            |   2360 characters


## 3. (Optional) NLP features: keywords & summary

`newspaper` can produce **keywords** and a **summary** via `article.nlp()`.
This feature needs `nltk` (the `punkt` data). Install:

```bash
uv sync --extra news-nlp
```

If `nltk` is not installed yet, the next cell will skip it safely.

## When to use what
- **Articles/news** (long text) → `newspaper4k` / `trafilatura` (automatic, multi-source).
- **E-commerce / tables / structured data** → BeautifulSoup, XPath, `pandas.read_html`, or an API.


In [3]:
# NLP features need nltk; only run if available
try:
    art.nlp()
    print("KEYWORDS:", art.keywords[:10])
    print("SUMMARY :", art.summary[:300], "...")
except Exception as e:
    print("Skipping NLP (nltk not installed / data not downloaded):", e)
    print("Install with: uv sync --extra news-nlp")


KEYWORDS: ['web', 'scraping', 'data', 'pages', 'edit', 'court', 'site', 'case', 'websites', 'access']
SUMMARY : Web scraping, web harvesting, or web data extraction is data scraping used for extracting data from websites.
[1] Web scraping software may directly access the World Wide Web using the Hypertext Transfer Protocol or a web browser.
While web scraping can be done manually by a software user, the term  ...
